In [8]:
import os
import pandas as pd
import sys

from pathlib import Path
current_dir = Path.cwd().resolve()
project_root = str(current_dir.parent if current_dir.name == "notebooks" else current_dir)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [9]:
from src.core.Context import context
from src.urv.URV import URV
from src.recommendation.RecommendationEngine import RecommendationEngine
from src.matrix.Matrix import Metrix
import numpy as np

In [10]:
import pickle

behaviours = context.behaviours

vectors_dir = os.path.join(project_root, "vectors")
with open(os.path.join(vectors_dir, "represent_vectors.pkl"), "rb") as f:
    represented_vector = pickle.load(f)
    
urv = URV(represented_vector=represented_vector)

recommender = RecommendationEngine(represented_vector=represented_vector)

In [11]:
sample_user_id = "U13740"
evaluation_rows = context.impressions.loc[
    context.impressions["user_id"] == sample_user_id
]
print(f"Evaluating user {sample_user_id}: {len(evaluation_rows)} impressions")

Evaluating user U13740: 3 impressions


In [12]:
results = []

for behaviour in evaluation_rows.itertuples(index=False):
    user_vector = urv.getURV(
        history=behaviour.history,
        user_id=behaviour.user_id,
        impression_id=behaviour.impression_id,
    )
    scored_candidates = recommender.calculate_by_impress(
        user_vector=user_vector,
        impressed_list=[behaviour.candidates],
    )[0]
    metrics = Metrix(scored_candidates).evaluate()
    results.append({
        "impression_id": behaviour.impression_id,
        "user_id": behaviour.user_id,
        **metrics,
    })

In [13]:
print("AUC:", np.nanmean([r["AUC"] for r in results]))
print("MRR:", np.mean([r["MRR"] for r in results]))
print("nDCG@5:", np.mean([r["nDCG@5"] for r in results]))
print("nDCG@10:", np.mean([r["nDCG@10"] for r in results]))

AUC: 0.8295005807200929
MRR: 0.3737373737373737
nDCG@5: 0.3333333333333333
nDCG@10: 0.3333333333333333


In [14]:
results

[{'impression_id': 1,
  'user_id': 'U13740',
  'AUC': 1.0,
  'MRR': 1.0,
  'nDCG@5': np.float64(1.0),
  'nDCG@10': np.float64(1.0)},
 {'impression_id': 35263,
  'user_id': 'U13740',
  'AUC': 0.8885017421602788,
  'MRR': 0.030303030303030304,
  'nDCG@5': np.float64(0.0),
  'nDCG@10': np.float64(0.0)},
 {'impression_id': 154837,
  'user_id': 'U13740',
  'AUC': 0.6,
  'MRR': 0.09090909090909091,
  'nDCG@5': np.float64(0.0),
  'nDCG@10': np.float64(0.0)}]